In [ ]:
# Example 1: Basic Database Connection and Table Operations
from sqlalchemy import create_engine, Column, Integer, String, text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
import tempfile
import os

# Create an in-memory SQLite database for testing
engine = create_engine('sqlite:///:memory:', echo=False)
Base = declarative_base()

# Define a simple model
class User(Base):
    __tablename__ = 'users'
    id = Column(Integer, primary_key=True)
    name = Column(String(50))
    email = Column(String(100))

# Create tables
Base.metadata.create_all(engine)
print("Database and tables created successfully")

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

# Insert some data
users_data = [
    User(name='Alice', email='alice@example.com'),
    User(name='Bob', email='bob@example.com'),
    User(name='Charlie', email='charlie@example.com')
]

session.add_all(users_data)
session.commit()
print(f"Added {len(users_data)} users to database")

# Query data
all_users = session.query(User).all()
print(f"\nTotal users in database: {len(all_users)}")
for user in all_users:
    print(f"ID: {user.id}, Name: {user.name}, Email: {user.email}")

session.close()
print("Basic database connection and table operations complete")

In [ ]:
# Example 2: Advanced ORM Relationships and Queries
from sqlalchemy import create_engine, Column, Integer, String, DateTime, ForeignKey, func
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from datetime import datetime

# Create engine and base
engine = create_engine('sqlite:///:memory:', echo=False)
Base = declarative_base()

# Define models with relationships
class Author(Base):
    __tablename__ = 'authors'
    id = Column(Integer, primary_key=True)
    name = Column(String(100))
    books = relationship('Book', back_populates='author')

class Book(Base):
    __tablename__ = 'books'
    id = Column(Integer, primary_key=True)
    title = Column(String(200))
    published_date = Column(DateTime)
    author_id = Column(Integer, ForeignKey('authors.id'))
    author = relationship('Author', back_populates='books')

# Create tables
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

# Create authors and books
authors_data = [
    Author(name='J.K. Rowling'),
    Author(name='George Orwell'),
    Author(name='Agatha Christie')
]

session.add_all(authors_data)
session.flush()  # Get IDs without committing

books_data = [
    Book(title='Harry Potter', published_date=datetime(1997, 6, 26), author_id=authors_data[0].id),
    Book(title='1984', published_date=datetime(1949, 6, 8), author_id=authors_data[1].id),
    Book(title='Animal Farm', published_date=datetime(1945, 8, 17), author_id=authors_data[1].id),
    Book(title='Murder on the Orient Express', published_date=datetime(1934, 1, 1), author_id=authors_data[2].id)
]

session.add_all(books_data)
session.commit()

# Advanced queries
print("Advanced ORM queries:")

# Join query with filtering
modern_books = session.query(Book).filter(Book.published_date > datetime(1950, 1, 1)).all()
print(f"\nBooks published after 1950: {len(modern_books)}")
for book in modern_books:
    print(f"  {book.title} by {book.author.name} ({book.published_date.year})")

# Aggregate query
author_book_counts = session.query(
    Author.name, 
    func.count(Book.id).label('book_count')
).join(Book).group_by(Author.id).all()

print("\nBooks per author:")
for author_name, book_count in author_book_counts:
    print(f"  {author_name}: {book_count} books")

session.close()
print("Advanced ORM relationships and queries complete")

In [ ]:
# Example 3: Raw SQL Queries and Connection Management
from sqlalchemy import create_engine, text, MetaData, Table, Column, Integer, String
from sqlalchemy.pool import StaticPool
import json

# Create engine with connection pooling
engine = create_engine(
    'sqlite:///:memory:', 
    echo=False,
    poolclass=StaticPool,
    pool_pre_ping=True
)

# Create table using Core API
metadata = MetaData()
products_table = Table(
    'products', metadata,
    Column('id', Integer, primary_key=True),
    Column('name', String(100)),
    Column('price', Integer),  # Store price in cents
    Column('category', String(50))
)

metadata.create_all(engine)

# Insert data using raw SQL
with engine.connect() as conn:
    # Insert sample products
    products_data = [
        {'name': 'Laptop', 'price': 99999, 'category': 'Electronics'},
        {'name': 'Book', 'price': 1999, 'category': 'Education'},
        {'name': 'Coffee Mug', 'price': 1299, 'category': 'Kitchen'},
        {'name': 'Smartphone', 'price': 79999, 'category': 'Electronics'},
        {'name': 'Notebook', 'price': 599, 'category': 'Education'}
    ]
    
    for product in products_data:
        conn.execute(
            text("INSERT INTO products (name, price, category) VALUES (:name, :price, :category)"),
            product
        )
    
    conn.commit()
    print(f"Inserted {len(products_data)} products")

# Complex queries using raw SQL
with engine.connect() as conn:
    # Price analysis query
    price_analysis = conn.execute(text("""
        SELECT 
            category,
            COUNT(*) as product_count,
            AVG(price) as avg_price,
            MIN(price) as min_price,
            MAX(price) as max_price
        FROM products 
        GROUP BY category
        ORDER BY avg_price DESC
    """)).fetchall()
    
    print("\nPrice analysis by category:")
    for row in price_analysis:
        avg_price_dollars = row.avg_price / 100
        min_price_dollars = row.min_price / 100
        max_price_dollars = row.max_price / 100
        print(f"  {row.category}: {row.product_count} items, "
              f"Avg: ${avg_price_dollars:.2f}, "
              f"Range: ${min_price_dollars:.2f}-${max_price_dollars:.2f}")
    
    # Parameterized query with user input simulation
    max_price_filter = 50000  # $500.00 in cents
    affordable_products = conn.execute(
        text("SELECT name, price, category FROM products WHERE price <= :max_price ORDER BY price"),
        {'max_price': max_price_filter}
    ).fetchall()
    
    print(f"\nProducts under ${max_price_filter/100:.2f}:")
    for product in affordable_products:
        price_dollars = product.price / 100
        print(f"  {product.name} (${price_dollars:.2f}) - {product.category}")

print("Raw SQL queries and connection management complete")

In [ ]:
# Example 4: Database Migrations and Schema Evolution
from sqlalchemy import create_engine, Column, Integer, String, Boolean, DateTime, text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from datetime import datetime
import tempfile
import os

# Create a temporary file database for migration demo
temp_db = tempfile.NamedTemporaryFile(delete=False, suffix='.db')
temp_db.close()
db_url = f'sqlite:///{temp_db.name}'

engine = create_engine(db_url, echo=False)
Base = declarative_base()

# Version 1 schema
class UserV1(Base):
    __tablename__ = 'users'
    id = Column(Integer, primary_key=True)
    username = Column(String(50))
    email = Column(String(100))

print("Creating initial schema (Version 1)...")
Base.metadata.create_all(engine)

# Insert initial data
Session = sessionmaker(bind=engine)
session = Session()

initial_users = [
    UserV1(username='alice', email='alice@example.com'),
    UserV1(username='bob', email='bob@example.com')
]
session.add_all(initial_users)
session.commit()
print(f"Added {len(initial_users)} users to V1 schema")

# Simulate schema migration
print("\nPerforming schema migration...")

with engine.connect() as conn:
    # Add new columns (simulating migration)
    try:
        conn.execute(text('ALTER TABLE users ADD COLUMN is_active BOOLEAN DEFAULT 1'))
        conn.execute(text('ALTER TABLE users ADD COLUMN created_at TIMESTAMP'))
        conn.commit()
        print("Added new columns: is_active, created_at")
    except Exception as e:
        print(f"Migration note: {e}")
        raise
    
    # Update existing records with default values
    current_time = datetime.now().isoformat()
    conn.execute(text(f"UPDATE users SET created_at = '{current_time}' WHERE created_at IS NULL"))
    conn.commit()
    print("Updated existing records with default values")

# Version 2 schema (with new fields)
class UserV2(Base):
    __tablename__ = 'users'
    __table_args__ = {'extend_existing': True}
    
    id = Column(Integer, primary_key=True)
    username = Column(String(50))
    email = Column(String(100))
    is_active = Column(Boolean, default=True)
    created_at = Column(DateTime, default=datetime.utcnow)

# Test the migrated schema
print("\nTesting migrated schema...")
session.close()
session = Session()

# Add new user with V2 schema
new_user = UserV2(username='charlie', email='charlie@example.com', is_active=True)
session.add(new_user)
session.commit()

# Query all users with new schema
all_users_v2 = session.query(UserV2).all()
print(f"\nAll users after migration ({len(all_users_v2)} total):")
for user in all_users_v2:
    status = "Active" if user.is_active else "Inactive"
    created = user.created_at.strftime('%Y-%m-%d %H:%M:%S') if user.created_at else "Unknown"
    print(f"  {user.username} ({user.email}) - {status}, Created: {created}")

session.close()

# Clean up
os.unlink(temp_db.name)
print("Database migrations and schema evolution complete")

In [ ]:
# Example 5: Production Database Patterns and Performance Optimization
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Index, func, event
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, load_only
from sqlalchemy.pool import QueuePool
from datetime import datetime, timedelta
import time
import random

# Production-style engine configuration
engine = create_engine(
    'sqlite:///:memory:',
    echo=False,  # Set to True for SQL debugging
    poolclass=QueuePool,
    pool_size=10,
    max_overflow=20,
    pool_pre_ping=True,
    pool_recycle=3600  # Recycle connections after 1 hour
)

Base = declarative_base()

# Optimized model with indexes
class LogEntry(Base):
    __tablename__ = 'log_entries'
    
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, index=True)  # Indexed for faster queries
    action = Column(String(100), index=True)
    timestamp = Column(DateTime, default=datetime.utcnow, index=True)
    ip_address = Column(String(45))
    user_agent = Column(String(500))
    
    # Composite index for common query patterns
    __table_args__ = (
        Index('idx_user_timestamp', 'user_id', 'timestamp'),
        Index('idx_action_timestamp', 'action', 'timestamp'),
    )

Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)

# Performance monitoring
query_stats = {'total_queries': 0, 'total_time': 0}

@event.listens_for(engine, 'before_cursor_execute')
def receive_before_cursor_execute(conn, cursor, statement, parameters, context, executemany):
    context._query_start_time = time.time()

@event.listens_for(engine, 'after_cursor_execute')
def receive_after_cursor_execute(conn, cursor, statement, parameters, context, executemany):
    total_time = time.time() - context._query_start_time
    query_stats['total_queries'] += 1
    query_stats['total_time'] += total_time

# Generate sample data efficiently
print("Generating sample log data...")
session = Session()

actions = ['login', 'logout', 'view_page', 'edit_profile', 'upload_file', 'download_file']
ip_addresses = ['192.168.1.10', '10.0.0.5', '172.16.0.20', '203.0.113.15']
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'
]

# Bulk insert for better performance
log_entries = []
for i in range(1000):
    entry = LogEntry(
        user_id=random.randint(1, 100),
        action=random.choice(actions),
        timestamp=datetime.utcnow() - timedelta(
            days=random.randint(0, 30),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        ),
        ip_address=random.choice(ip_addresses),
        user_agent=random.choice(user_agents)
    )
    log_entries.append(entry)

session.bulk_save_objects(log_entries)
session.commit()
print(f"Inserted {len(log_entries)} log entries")

# Optimized queries with performance analysis
print("\nPerformance-optimized queries:")

start_time = time.time()

# Query 1: Recent activity with projection (load only needed columns)
# Note: load_only() requires actual column attributes, not strings
recent_activities = session.query(LogEntry)\
    .options(load_only(LogEntry.action, LogEntry.timestamp, LogEntry.user_id))\
    .filter(LogEntry.timestamp > datetime.utcnow() - timedelta(days=7))\
    .order_by(LogEntry.timestamp.desc())\
    .limit(10)\
    .all()

print(f"Recent activities (last 7 days): {len(recent_activities)} entries")
for entry in recent_activities[:3]:
    print(f"  User {entry.user_id}: {entry.action} at {entry.timestamp.strftime('%Y-%m-%d %H:%M')}")

# Query 2: Aggregated statistics
action_stats = session.query(
    LogEntry.action,
    func.count(LogEntry.id).label('count'),
    func.count(func.distinct(LogEntry.user_id)).label('unique_users')
).group_by(LogEntry.action).order_by(func.count(LogEntry.id).desc()).all()

print("\nAction statistics:")
for action, count, unique_users in action_stats:
    print(f"  {action}: {count} total, {unique_users} unique users")

# Query 3: User activity analysis with window functions (SQLite limitation workaround)
active_users = session.query(
    LogEntry.user_id,
    func.count(LogEntry.id).label('activity_count'),
    func.max(LogEntry.timestamp).label('last_activity')
).group_by(LogEntry.user_id)\
 .having(func.count(LogEntry.id) > 5)\
 .order_by(func.count(LogEntry.id).desc())\
 .limit(5).all()

print("\nMost active users:")
for user_id, activity_count, last_activity in active_users:
    print(f"  User {user_id}: {activity_count} actions, last seen {last_activity.strftime('%Y-%m-%d %H:%M')}")

total_time = time.time() - start_time

# Performance summary
print(f"\nPerformance Summary:")
print(f"Total queries executed: {query_stats['total_queries']}")
print(f"Total query time: {query_stats['total_time']:.3f}s")
print(f"Average query time: {query_stats['total_time']/query_stats['total_queries']:.3f}s")
print(f"Overall execution time: {total_time:.3f}s")

# Connection pool status
pool = engine.pool
print(f"Connection pool - Size: {pool.size()}, Checked out: {pool.checkedout()}, Overflow: {pool.overflow()}")

session.close()
print("Production database patterns and performance optimization complete")